<div style="background: linear-gradient(135deg, rgba(30, 30, 30, 0.8), rgba(50, 50, 50, 0.9)); padding: 25px; border-radius: 12px; box-shadow: 0 4px 8px rgba(255, 255, 255, 0.1);">
    <h1  style="font-size: 28px; font-family: 'Segoe UI', sans-serif; color: #00c8ff; text-align: center; font-weight: bold;"><b>1. 🚘 License Plate Detection using YOLOv8</b></h1>

<p style="font-size: 18px; font-family: 'Segoe UI', sans-serif; color: #ddd; text-align: center;">
        A deep learning approach for real-time vehicle license plate detection and recognition. 
        Powered by <strong>YOLOv8</strong> and optimized for speed and accuracy. 🚀
    </p>

<p style="font-size: 22px; font-family: 'Segoe UI', sans-serif; color: #00c8ff; text-align: center; margin-top: 20px;">
        🔹 Why This Project?
    </p>

<ul style="font-size: 18px; font-family: 'Segoe UI', sans-serif; color: #ccc; line-height: 1.6;">
        <li>⚡ <strong>Fast & Efficient:</strong> YOLOv8 enables real-time plate detection.</li>
        <li>🎯 <strong>High Accuracy:</strong> Optimized model for precise recognition.</li>
        <li>🚗 <strong>Smart Applications:</strong> Useful for traffic monitoring, toll collection, and security.</li>
    </ul>

<p style="font-size: 22px; font-family: 'Segoe UI', sans-serif; color: #00c8ff; text-align: center;">
        📊 Let's Dive into the Code! 🔥
    </p>
</div>


# 2. 🔹 Why YOLOv8n for License Plate Detection?

<div style="background: linear-gradient(135deg, rgba(30, 30, 30, 0.8), rgba(50, 50, 50, 0.9)); padding: 25px; border-radius: 12px; box-shadow: 0 4px 8px rgba(255, 255, 255, 0.1);">

<ul style="font-size: 18px; font-family: 'Segoe UI', sans-serif; color: #ccc; line-height: 1.6;">
        <li>
            <strong>Lightweight & Fast</strong><br>
            YOLOv8n (nano) is the smallest variant in the YOLOv8 family.<br>
            Its reduced size makes it faster on real-time systems (e.g. traffic cameras),
            which is crucial for license plate detection where speed matters.
        </li>
        <li>
            <strong>High Accuracy on Small Objects</strong><br>
            License plates are relatively small and often at varying angles.
            YOLOv8n's improved architecture and anchor-free approach help detect
            small targets more reliably than older YOLO versions (v3, v4, v5, v7).
        </li>
        <li>
            <strong>Comparison with “YOLOv11” or Future Versions</strong><br>
            While upcoming versions may promise higher accuracy, they often come with
            larger model sizes or higher computation needs. YOLOv8n strikes a balance
            between performance and resource usage, ideal for embedded devices or low-end GPUs.
        </li>
        <li>
            <strong>Community Support & Documentation</strong><br>
            YOLOv8 is officially released by Ultralytics, ensuring solid documentation
            and community-driven improvements. Experimental or hypothetical releases
            might lack maturity and broad support.
        </li>
    </ul>
    <p style="font-size: 18px; font-family: 'Segoe UI', sans-serif; color: #ddd; text-align: left; margin-top: 20px;">
        Overall, YOLOv8n provides a sweet spot of speed, accuracy, 
        and smaller memory footprint—perfect for real-time license plate detection.
    </p>
</div>


-----------------


# 3. 🔹 Installing & Importing Libraries
<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #00c8ff; border-radius: 8px;">
<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffd700; font-size: 16px; font-family: 'Segoe UI', sans-serif; white-space: pre-wrap;">
# Install required libraries
        <br>!pip install ultralytics
        <br>!pip install albumentations
    </p>
</div>


In [ ]:
%%capture
!pip install ultralytics
# !pip install albumentations

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Import necessary libraries
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import cv2
import yaml
import torch
from PIL import Image
from IPython.display import display, Video
from ultralytics import YOLO
from collections import deque
import subprocess  # For executing ffmpeg

# 4. 🔹 Why `data.yaml`?

<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #00c8ff; border-radius: 8px;">

<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffd700; font-size: 16px; font-family: 'Segoe UI', sans-serif; white-space: pre-wrap;">
        The <code>data.yaml</code> file is used by YOLO to:
        <br>✅ Specify dataset paths (<strong>train</strong>, <strong>val</strong>, <strong>test</strong>).
        <br>✅ Define the number of classes (<strong>nc</strong>) and their names.
        <br>📌 This helps YOLO know where to find your data and how to handle it during training.
    </p>
</div>


In [ ]:
# Define dataset paths
train_dir = "/kaggle/input/license-plate-dataset/archive/images/train"
val_dir = "/kaggle/input/license-plate-dataset/archive/images/val"

# Define class names
classes = ["license_plate"]  # Add more classes if needed

# Create YAML content
data_yaml_content = f"""
train: {train_dir}
val: {val_dir}

nc: {len(classes)}  # Number of classes
names: {classes}  # Class names
"""

# Save to a file
yaml_path = "/kaggle/working/data.yaml"
with open(yaml_path, "w") as f:
    f.write(data_yaml_content)

# Display the content for verification
!cat {yaml_path}


# 5. 🔹 What is YOLOv8?

<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #00c8ff; border-radius: 8px;">

<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffd700; font-size: 16px; font-family: 'Segoe UI', sans-serif; white-space: pre-wrap;">
YOLOv8 is the latest version of the YOLO family by Ultralytics,
        <br>✅ Faster and more accurate object detection.
        <br>✅ Anchor-free design for better small object recognition.
        <br>✅ Easy deployment in various environments.
        <br>📌 Its balance of speed and precision makes it ideal for tasks like license plate detection, surveillance, and autonomous driving.
    </p>
</div>


In [ ]:
model = YOLO("yolov8n.pt")

# 6. 🚀 Training the Model

<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #ffcc00; border-radius: 8px; margin-top: 20px;">

<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffcc00; font-size: 16px;">
        Fine-Tuning YOLOv8 on License Plate Detection Dataset...
    </p>
</div>


In [ ]:
results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,  
    imgsz=640, 
    lr0= 0.0005,  
    batch=32,
    lrf=0.1,
    augment=True
)

# 7. 📊 Training Results

<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #00ff99; border-radius: 8px; margin-top: 20px;">

<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffd700; font-size: 16px; font-family: 'Segoe UI', sans-serif; white-space: pre-wrap;">
        Visualizing the performance of YOLOv8 after training...
    </p>
</div>


In [ ]:
train_results = "/kaggle/working/runs/detect/train/results.png"

if os.path.exists(train_results):
    img = Image.open(train_results)  # Use PIL to read the image
    display(img)  # Display the image
else:
    print("Training results image not found.")



In [ ]:
map_column = "metrics/mAP50(B)" 
loss_column_train = "train/box_loss"
loss_column_val = "val/box_loss"

plt.figure(figsize=(12, 5))
sns.set_style("whitegrid")

# 📌 Plot Train Loss and Validation Loss
plt.subplot(1, 2, 1)
plt.plot(df["epoch"], df[loss_column_train], label="Train Box Loss", color="red")
plt.plot(df["epoch"], df[loss_column_val], label="Validation Box Loss", color="blue")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()

# 📌 Plot mAP@50
plt.subplot(1, 2, 2)
plt.plot(df["epoch"], df[map_column], label="mAP@50", color="green")
plt.xlabel("Epochs")
plt.ylabel("mAP")
plt.title("Mean Average Precision (mAP)")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Collect all images from the directory
all_images = glob.glob("/kaggle/input/number-plate-detection/images/*.jpeg")

# Randomly select 6 different images each time
test_images = random.sample(all_images, 6)

# Create a figure with 2 rows and 3 columns
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for img_path, ax in zip(test_images, axes.flatten()):
    results = model(img_path)      # Run YOLO on the image
    result_img = results[0].plot() # Get the annotated image
    
    # Convert the annotated image to RGB for plotting
    img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.axis("off")
    ax.set_title(os.path.basename(img_path))  # Show the image filename

plt.tight_layout()
plt.show()

# 8. 🎥 Training & Evaluating on Video

<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #00c8ff; border-radius: 8px;">

<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffd700; font-size: 16px; font-family: 'Segoe UI', sans-serif; white-space: pre-wrap;">
        In this section, we use YOLO to process a video, 
        apply the trained model, and visualize the results.
    </p>
</div>


In [ ]:
# Load the trained YOLOv8 model
model_path = "/kaggle/working/runs/detect/train/weights/best.pt"
model = YOLO(model_path)

# Define input and output video paths
input_video = "/kaggle/input/car-number-plate-video/Automatic Number Plate Recognition (ANPR) _ Vehicle Number Plate Recognition (1).mp4"
output_video = "/kaggle/working/output_video.mp4"
compressed_video = "/kaggle/working/output_video_compressed.mp4"  # Compressed version

# Open the input video file
cap = cv2.VideoCapture(input_video)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# Process video frame by frame
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 on the frame
    results = model(frame, verbose=False)  # No extra logs

    # Draw bounding boxes
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf[0].item()
            label = f"Plate {conf:.2f}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Write frame to output video
    out.write(frame)
    frame_count += 1

# Release video resources
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"✅ Detection completed! Processed {frame_count} frames.")

# ✅ **Compress the video using FFmpeg**
print("⏳ Compressing video... (This may take a few seconds)")
ffmpeg_command = [
    "ffmpeg", "-i", output_video, "-vcodec", "libx264", "-crf", "28", "-preset", "fast", compressed_video
]
subprocess.run(ffmpeg_command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("✅ Compression done!")

# Function to display video in Jupyter Notebook (Kaggle Compatible)
from IPython.display import HTML
from base64 import b64encode

def play_video(file_path, width=800):
    """Display a video file directly in a Jupyter Notebook (Kaggle compatible)."""
    try:
        with open(file_path, "rb") as video_file:
            video_data = video_file.read()
            video_base64 = b64encode(video_data).decode()
    except FileNotFoundError:
        return HTML("<p style='color:red;'>🚨 Error: Video file not found!</p>")
    
    video_html = f"""
    <video width="{width}" controls autoplay loop>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """
    
    return HTML(video_html)

# Display the compressed video
play_video(compressed_video, width=1000)


-----------------

# 9. 🙏 Thank You for Your Support!

<div style="background: rgba(20, 20, 20, 0.95); padding: 20px; border-left: 5px solid #ff9800; border-radius: 8px;">

<p style="background: #1e1e1e; padding: 12px; border-radius: 5px; color: #ffd700; font-size: 16px; font-family: 'Segoe UI', sans-serif; white-space: pre-wrap;">
🚀 I appreciate you taking the time to review my work!  
Your feedback is invaluable, and I'm always looking for ways to improve.  
💡 If you found this notebook helpful, an upvote would mean a lot!  
It helps me continue creating valuable content.  
💬 Feel free to leave any suggestions, questions, or improvements in the comments.  
Happy coding! 😊
</p>
</div>
